In [ ]:
import pandas as pd
import numpy as np
import re

In [ ]:
from google.colab import files
uploaded=files.upload()

In [ ]:
import io
df= pd.read_excel(io.BytesIO(uploaded['merged_90_c1 (1) (1).xlsx']))

In [ ]:
df = df.drop(['patient_id'], axis=1)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

X = df.drop('dialysis', axis=1)

y = df['dialysis']



X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
train_df = pd.concat([X_train, y_train], axis=1)

In [ ]:
train_df.shape

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import norm
import warnings

warnings.filterwarnings("ignore")


np.random.seed(0)


minority_data = train_df[train_df['dialysis'] == 1].drop(columns=['dialysis'])

marginal_probs = minority_data.mean().values


cor_matrix = minority_data.corr().values


def generate_gaussian_copula(cor_matrix, num_samples, marginal_probs):
    num_vars = cor_matrix.shape[0]
    mean = np.zeros(num_vars)
    mvn_samples = np.random.multivariate_normal(mean, cor_matrix, num_samples)
    uniform_samples = norm.cdf(mvn_samples)
    synthetic_data = (uniform_samples < marginal_probs).astype(int)
    return synthetic_data

num_synthetic_samples = 2407


synthetic_datasets = []
synthetic_datasets = {}

for i in range(1, 26):
    synthetic_minority_samples = generate_gaussian_copula(cor_matrix, num_synthetic_samples, marginal_probs)
    synthetic_minority_df = pd.DataFrame(synthetic_minority_samples, columns=minority_data.columns)
    synthetic_minority_df['dialysis'] = 1
    balanced_df = pd.concat([train_df, synthetic_minority_df], ignore_index=True)

    # Dynamically create a variable like balanced_df_1, balanced_df_2, ..., balanced_df_25
    globals()[f'balanced_df_{i}'] = balanced_df




In [ ]:
balanced_df_22.shape

Balanced_df_1 to balanced_df_25 are the datasets that the model has generated. We try 25 runs to eliminate the effect of randomness. For each, LLM should do the filtering. Please put your code below, you may save them in the same names (balanced_df_1, ...., balanced_df_25), so you can use the code for the prediction below.

In [ ]:
#put your llm code here habibi!











In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
import matplotlib.pyplot as plt

metrics = ['recall', 'precision', 'f1-score', 'accuracy']
classifier_names = ['DecisionTree', 'RandomForest', 'SVM', 'LogisticRegression']
results = {clf: {metric: [] for metric in metrics} for clf in classifier_names}
recall_scores_class_1 = {clf: [] for clf in classifier_names}

# Loop through each of the 25 DataFrames: balanced_df_1 to balanced_df_25
for i in range(1, 26):
    balanced_df = globals()[f'balanced_df_{i}']
    X = balanced_df.drop('dialysis', axis=1)
    y = balanced_df['dialysis']

    classifiers = {
        'DecisionTree': DecisionTreeClassifier(random_state=42),
        'RandomForest': RandomForestClassifier(random_state=42),
        'SVM': SVC(random_state=42),
        'LogisticRegression': LogisticRegression(random_state=42)
    }

    for name, clf in classifiers.items():
        clf.fit(X, y)
        y_pred = clf.predict(X_test)

        report = classification_report(y_test, y_pred, output_dict=True)
        results[name]['recall'].append(report['weighted avg']['recall'])
        results[name]['precision'].append(report['weighted avg']['precision'])
        results[name]['f1-score'].append(report['weighted avg']['f1-score'])
        results[name]['accuracy'].append(accuracy_score(y_test, y_pred))
        recall_scores_class_1[name].append(report['1']['recall'])


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

data = [recall_scores_class_1['DecisionTree'],
        recall_scores_class_1['RandomForest'],
        recall_scores_class_1['SVM'],
        recall_scores_class_1['LogisticRegression']]

ax.boxplot(data)
ax.set_xticklabels(['Decision Tree', 'Random Forest', 'SVM', 'Logistic Regression'])
ax.set_ylabel('Recall Score')
ax.set_xlabel('Classifiers')
ax.set_title('Class 1 Recall Scores Across 25 Balanced Datasets')
ax.grid(True, linestyle='-', alpha=0.7)

plt.savefig("recall_class1_Binary_copula.pdf", dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# code is after this.

In [ ]:
from transformers import GPT2Model, GPT2Tokenizer, DataCollatorWithPadding, Trainer, TrainingArguments
import torch.nn as nn
import torch
from torch.utils.data import Dataset
from Refined_dataset import call_dataset_infr
import evaluate
import numpy as np
from safetensors.torch import load_file

class TabularDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val) for key, val in self.encodings[idx].items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.float)
        return item

    def __len__(self):
        return len(self.labels)
class GPT2ForBinaryClassification(nn.Module):
    def __init__(self, model_name='gpt2', num_labels=1):
        super(GPT2ForBinaryClassification, self).__init__()
        self.gpt2 = GPT2Model.from_pretrained(model_name)
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(self.gpt2.config.hidden_size, num_labels)

    def forward(self, input_ids, attention_mask=None, labels=None):
        outputs = self.gpt2(input_ids, attention_mask=attention_mask)
        # Use the last hidden state of the last token
        last_hidden_state = outputs.last_hidden_state
        pooled_output = last_hidden_state[:, -1, :]
        pooled_output = self.dropout(pooled_output)
        logits = self.classifier(pooled_output)

        loss = None
        if labels is not None:
            loss_fct = nn.BCEWithLogitsLoss()
            loss = loss_fct(logits, labels.view(-1, 1).to(dtype=torch.float32))

        return {'loss': loss, 'logits': logits}
def serialize_row(row):
    features = ', '.join([f"feature_{i+1}: {row[i]}" for i in range(len(row)-1)])
    label = row[-1]
    return f"{features}. Label: {label}."

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return accuracy.compute(predictions=predictions, references=labels)


# Initialize tokenizer
# Set the path where your checkpoint is stored
model_path = "./CTGAN/results/checkpoint-2165"  # <- update to your actual folder

# Load tokenizer
tokenizer = GPT2Tokenizer.from_pretrained(model_path)

# tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
tokenizer.padding_side = "left"
tokenizer.pad_token = tokenizer.eos_token

# Set the pad_token to eos_token to avoid padding issues
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Tokenize the inputs
def preprocess_function(examples):
    return tokenizer(examples["text"], truncation=True, padding=True, max_length=512, return_tensors='pt')

accuracy = evaluate.load("accuracy")

# Initialize the model
model = GPT2ForBinaryClassification()

# Step 2: Load the safetensors weights
state_dict = load_file("./CTGAN/results/checkpoint-2165/model.safetensors")  # adjust path
model.load_state_dict(state_dict)
# Inference Code
from torch.utils.data import DataLoader, Dataset

class InferenceDataset(Dataset):
    def __init__(self, texts):
        self.texts = texts

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        return self.texts[idx]

def predict_from_dataframe(df, model, tokenizer, device='cuda', batch_size=128):
    model.eval()
    model.to(device)

    dataset = InferenceDataset(df.tolist())
    dataloader = DataLoader(dataset, batch_size=batch_size)

    all_preds = []
    all_probs = []

    with torch.no_grad():
        for batch in dataloader:
            inputs = tokenizer(batch, return_tensors='pt', padding=True, truncation=True, max_length=512)
            inputs = {k: v.to(device) for k, v in inputs.items()}

            outputs = model(**inputs)
            logits = outputs['logits']
            probs = torch.sigmoid(logits)
            preds = (probs > 0.5).long()

            all_preds.append(preds.cpu())
            all_probs.append(probs.cpu())

    # Concatenate all batches
    all_preds = torch.cat(all_preds).numpy()
    all_probs = torch.cat(all_probs).numpy()

    # Count number of predictions that are 1
    num_positive_preds = (all_preds == 1).sum()

    print(f"Number of positive predictions (label = 1): {num_positive_preds}")

    return all_preds, all_probs
def predict(texts, model, tokenizer, device='cuda'):
    model.eval()
    model.to(device)

    # Tokenize input texts
    inputs = tokenizer(texts.tolist(), return_tensors='pt', padding=True, truncation=True, max_length=512)
    inputs = {key: val.to(device) for key, val in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs['logits']
        probs = torch.sigmoid(logits)  # Convert logits to probabilities for binary classification
        preds = (probs > 0.5).long()   # Thresholding at 0.5
    return preds, probs




# #Example inference with the test data
# test_dataset = tokenized_data["test"]
# test_texts = [tokenizer.decode(test_dataset[i]['input_ids'], skip_special_tokens=True) for i in range(len(test_dataset))]
# for i, text in enumerate(test_texts[0:5]): # just doing the first 5 for example
#     predicted_probability = predict(text, model, tokenizer)
#     print(f"Text: {text}, Predicted Probability: {predicted_probability}")













"""BGCS+LLM.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1KwmdjIe1lqly-SSF2732YJ3Gmrj6jm_1
"""

import pandas as pd

df = pd.read_excel('./CTGAN/dataset/merged_90_c1.xlsx')

df = df.drop(['patient_id'], axis=1)

from sklearn.model_selection import train_test_split


X = df.drop('dialysis', axis=1)

y = df['dialysis']



X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

train_df = pd.concat([X_train, y_train], axis=1)

from scipy.stats import norm
import warnings

warnings.filterwarnings("ignore")


np.random.seed(0)


minority_data = train_df[train_df['dialysis'] == 1].drop(columns=['dialysis'])

marginal_probs = minority_data.mean().values


cor_matrix = minority_data.corr().values


def generate_gaussian_copula(cor_matrix, num_samples, marginal_probs):
    num_vars = cor_matrix.shape[0]
    mean = np.zeros(num_vars)
    mvn_samples = np.random.multivariate_normal(mean, cor_matrix, num_samples)
    uniform_samples = norm.cdf(mvn_samples)
    synthetic_data = (uniform_samples < marginal_probs).astype(int)
    return synthetic_data

num_synthetic_samples = 5000


synthetic_datasets = []
synthetic_datasets = {}

for i in range(1, 26):
    synthetic_minority_samples = generate_gaussian_copula(cor_matrix, num_synthetic_samples, marginal_probs)
    synthetic_minority_df = pd.DataFrame(synthetic_minority_samples, columns=minority_data.columns)
    data = call_dataset_infr(synthetic_minority_df)
    preds, _ =  predict_from_dataframe(data, model, tokenizer)
    synthetic_minority_df['dialysis'] = preds
    # Keep only rows where predicted_label == 1
    synthetic_minority_df = synthetic_minority_df[synthetic_minority_df['dialysis'] == 0].reset_index(drop=True)
    synthetic_minority_df = synthetic_minority_df.drop(['text'], axis=1)
    synthetic_minority_df['dialysis'] = 1
    balanced_df = pd.concat([train_df, synthetic_minority_df], ignore_index=True)

    # Dynamically create a variable like balanced_df_1, balanced_df_2, ..., balanced_df_25
    globals()[f'balanced_df_{i}'] = balanced_df


"""Balanced_df_1 to balanced_df_25 are the datasets that the model has generated. We try 25 runs to eliminate the effect of randomness. For each, LLM should do the filtering. Please put your code below, you may save them in the same names (balanced_df_1, ...., balanced_df_25), so you can use the code for the prediction below."""

#put your llm code here habibi!


from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
import matplotlib.pyplot as plt

metrics = ['recall', 'precision', 'f1-score', 'accuracy']
classifier_names = ['DecisionTree', 'RandomForest', 'SVM', 'LogisticRegression']
results = {clf: {metric: [] for metric in metrics} for clf in classifier_names}
recall_scores_class_1 = {clf: [] for clf in classifier_names}

# Loop through each of the 25 DataFrames: balanced_df_1 to balanced_df_25
for i in range(1, 26):
    balanced_df = globals()[f'balanced_df_{i}']
    X = balanced_df.drop('dialysis', axis=1)
    y = balanced_df['dialysis']

    classifiers = {
        'DecisionTree': DecisionTreeClassifier(random_state=42),
        'RandomForest': RandomForestClassifier(random_state=42),
        'SVM': SVC(random_state=42),
        'LogisticRegression': LogisticRegression(random_state=42)
    }

    for name, clf in classifiers.items():
        clf.fit(X, y)
        y_pred = clf.predict(X_test)

        report = classification_report(y_test, y_pred, output_dict=True)
        results[name]['recall'].append(report['weighted avg']['recall'])
        results[name]['precision'].append(report['weighted avg']['precision'])
        results[name]['f1-score'].append(report['weighted avg']['f1-score'])
        results[name]['accuracy'].append(accuracy_score(y_test, y_pred))
        recall_scores_class_1[name].append(report['1']['recall'])

fig, ax = plt.subplots(figsize=(10, 6))

data = [recall_scores_class_1['DecisionTree'],
        recall_scores_class_1['RandomForest'],
        recall_scores_class_1['SVM'],
        recall_scores_class_1['LogisticRegression']]

ax.boxplot(data)
ax.set_xticklabels(['Decision Tree', 'Random Forest', 'SVM', 'Logistic Regression'])
ax.set_ylabel('Recall Score')
ax.set_xlabel('Classifiers')
ax.set_title('Class 1 Recall Scores Across 25 Balanced Datasets')
ax.grid(True, linestyle='-', alpha=0.7)

plt.savefig("recall_class1_Binary_copula.pdf", dpi=300, bbox_inches='tight')
# plt.show()


In [ ]:
# train code:

In [ ]:
from transformers import GPT2Model, GPT2Tokenizer, DataCollatorWithPadding, Trainer, TrainingArguments
import torch.nn as nn
import torch
from torch.utils.data import Dataset
from Refined_dataset import call_dataset
import evaluate
import numpy as np

class TabularDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val) for key, val in self.encodings[idx].items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.float)
        return item

    def __len__(self):
        return len(self.labels)
class GPT2ForBinaryClassification(nn.Module):
    def __init__(self, model_name='gpt2', num_labels=1):
        super(GPT2ForBinaryClassification, self).__init__()
        self.gpt2 = GPT2Model.from_pretrained(model_name)
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(self.gpt2.config.hidden_size, num_labels)

    def forward(self, input_ids, attention_mask=None, labels=None):
        outputs = self.gpt2(input_ids, attention_mask=attention_mask)
        # Use the last hidden state of the last token
        last_hidden_state = outputs.last_hidden_state
        pooled_output = last_hidden_state[:, -1, :]
        pooled_output = self.dropout(pooled_output)
        logits = self.classifier(pooled_output)

        loss = None
        if labels is not None:
            loss_fct = nn.BCEWithLogitsLoss()
            loss = loss_fct(logits, labels.view(-1, 1).to(dtype=torch.float32))

        return {'loss': loss, 'logits': logits}
def serialize_row(row):
    features = ', '.join([f"feature_{i+1}: {row[i]}" for i in range(len(row)-1)])
    label = row[-1]
    return f"{features}. Label: {label}."

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return accuracy.compute(predictions=predictions, references=labels)


# Initialize tokenizer
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
tokenizer.padding_side = "left"
tokenizer.pad_token = tokenizer.eos_token

# Set the pad_token to eos_token to avoid padding issues
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Tokenize the inputs
def preprocess_function(examples):
    return tokenizer(examples["text"], truncation=True, padding=True, max_length=512, return_tensors='pt')

data = call_dataset()

tokenized_data = data.map(preprocess_function, batched=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

accuracy = evaluate.load("accuracy")

# Initialize the model
model = GPT2ForBinaryClassification()

# Define training arguments
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
)

# Initialize the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_data["train"],
    eval_dataset=tokenized_data["test"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# Fine-tune the model
trainer.train()
print('done!')

# Evaluate the model
eval_result = trainer.evaluate(eval_dataset=tokenized_data["test"])
print(f"Validation Loss: {eval_result['eval_loss']}")